# 01 — Los Angeles parking foundation inspection

Executed against real downloaded data. Unknown supply/occupancy stays unknown. This notebook does not train a citywide model.

In [1]:
from pathlib import Path
import json
import pandas as pd
import geopandas as gpd
import duckdb
ROOT=Path.cwd()
assert (ROOT/'data/processed').exists()
coverage=json.loads((ROOT/'reports/coverage.json').read_text())
pd.DataFrame([coverage['geography']])

,source_streets,city_segments,block_faces,pilot_segments,pilot_faces,authoritative_curb_faces,invalid_segments,invalid_faces,excluded_segments
0,85190,81315,162630,981,1962,84,0,0,0


## Jurisdiction and approximate curb sides

Official City boundary clips the street source. Offset faces are approximate. Physical curb footprints do not establish eligible capacity.

In [2]:
segments=gpd.read_parquet('data/processed/street_segments.parquet')
faces=gpd.read_parquet('data/processed/pilot_block_faces.parquet')
assert segments.segment_id.is_unique
assert segments.is_valid.all()
faces.groupby(['geometry_method','applicability']).size().to_frame('faces')

,,faces
geometry_method,applicability,
approximate_centerline_offset,needs_review,1878
authoritative_curb,needs_review,84


In [3]:
mapping=pd.read_parquet('data/processed/meter_face_mapping.parquet')
mapping.join_status.value_counts().to_frame('pilot_meter_joins')

,pilot_meter_joins
join_status,
matched,1562
ambiguous,22


## Time-specific evidence

Current states are last-known transactions. Age is measured at retrieval. Historical event-time states are not continuous verified ground truth. Persistence caps are sensitivity policies, and UNKNOWN remains separate.

In [4]:
pd.DataFrame(coverage['history'])

,month,raw_city_rows,pilot_rows,pilot_spaces,invalid_pilot_rows,duplicate_pilot_rows,min_utc,max_utc
0,2025_09,3586854,245830,337,0,0,2025-09-02 07:00:01+00:00,2025-10-01 02:59:38+00:00
1,2026_08,3409053,126313,140,0,0,2026-08-01 07:02:12+00:00,2026-09-01 06:58:59+00:00


In [5]:
current=pd.read_parquet('data/processed/pilot_current_occupancy.parquet')
current.groupby(['state','stale']).size().to_frame('meters')

meters
state    stale        
OCCUPIED False       2
         True       93
VACANT   False       3
         True       34

In [6]:
pd.read_parquet('data/processed/coverage_sensitivity.parquet')

,source_id,horizon_minutes,known_hours,expected_space_hours,known_fraction,unknown_state_rows,spaces
0,ladot_archive_2025_09,5,14113.604444,242640.0,0.058167,0,337
1,ladot_archive_2025_09,15,32163.310556,242640.0,0.132556,0,337
2,ladot_archive_2025_09,60,78443.943611,242640.0,0.323294,0,337
3,ladot_archive_2026_08,5,7443.692500,104160.0,0.071464,0,140
4,ladot_archive_2026_08,15,17109.566111,104160.0,0.164262,0,140
5,ladot_archive_2026_08,60,40788.053611,104160.0,0.391590,0,140


## Context and source quality

Counts measure the downloaded registry/layer. They are not occupancy labels. Actual observation dates can differ sharply from a file name or metadata update timestamp.

In [7]:
registry=json.loads(Path('config/source_registry.json').read_text())
pd.DataFrame(registry['sources'])[['id','status','priority','temporal_coverage']]

,id,status,priority,temporal_coverage
0,meters,verified/downloaded,1,See source_updated_at_utc and actual_observati...
1,occupancy,verified/downloaded,1,See source_updated_at_utc and actual_observati...
2,history,verified/downloaded,1,See source_updated_at_utc and actual_observati...
3,sweeping,stale,2,See source_updated_at_utc and actual_observati...
4,permits,stale,2,See source_updated_at_utc and actual_observati...
5,signs,stale,2,See source_updated_at_utc and actual_observati...
6,business,verified/downloaded,2,See source_updated_at_utc and actual_observati...
7,boundary,verified/downloaded,1,Current snapshot; per-feature creation/modific...
8,streets,verified/downloaded,1,Current snapshot; per-feature creation/modific...
9,curbs,verified/downloaded,1,Current snapshot; per-feature creation/modific...


In [8]:
pd.DataFrame(coverage['context_layers']).T

,features,invalid,empty
curbs,20877,3,0
driveways,5965,0,0
hydrants,763,0,0
regulatory_signs,907,0,0
bikeways,167,0,0
red_curb,12,0,0
buildings,10447,0,0
zoning,738,0,0
construction,3502,0,0
construction_streets,18,0,0


In [9]:
with duckdb.connect('data/processed/parking.duckdb',read_only=True) as con:
    summary=con.sql('SELECT count(*) AS faces, count(physical_capacity) AS faces_with_capacity FROM curb_inventory').df()
summary

,faces,faces_with_capacity
0,162630,0


In [10]:
from IPython.display import IFrame
IFrame('../reports/coverage_map.html',width='100%',height=600)

## Modeling gate

There are **zero verified unmetered occupancy labels**. Use the manual schema/workflow to collect them. Metered temporal experiments can begin after reviewing sensor freshness and historical inventory mapping; duration-specific legal suitability and citywide predictions are not validated. Use forward-time and held-out-face evaluation, with as-of features and prospective surveys.